# 02. CHIA DỮ LIỆU HUẤN LUYỆN USD/VND

Giữ nguyên thiết kế của bộ gốc:
- **Train:** từ đầu mẫu đến 31/12/2024
- **Validation:** toàn bộ năm 2025
- **Test:** từ 01/01/2026 đến hết dữ liệu hiện có


In [1]:
import pandas as pd
from pathlib import Path

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "code":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

PROCESSED_DIR = BASE_DIR / "data" / "processed"

print("PROCESSED_DIR:", PROCESSED_DIR)


PROCESSED_DIR: /mnt/data/USD_VND_notebooks/data/processed


In [2]:
# ============================================================
# 1. ĐỌC DATA GIÁ SẠCH
# ============================================================

price = pd.read_csv(
    PROCESSED_DIR / "usd_vnd_price_clean.csv",
    parse_dates=["Date"]
)

# ============================================================
# 2. ĐỌC DATA RETURN SẠCH
# ============================================================

returns = pd.read_csv(
    PROCESSED_DIR / "usd_vnd_return_clean.csv",
    parse_dates=["Date"]
)

price = price.sort_values("Date").reset_index(drop=True)
returns = returns.sort_values("Date").reset_index(drop=True)

print("Price:", price.shape, price["Date"].min(), "->", price["Date"].max())
print("Return:", returns.shape, returns["Date"].min(), "->", returns["Date"].max())


Price: (2784, 6) 2016-01-04 00:00:00 -> 2026-08-31 00:00:00
Return: (2783, 7) 2016-01-05 00:00:00 -> 2026-08-31 00:00:00


In [3]:
# ============================================================
# 3. CHIA TRAIN / VALIDATION / TEST
# ============================================================

train_end = pd.Timestamp("2024-12-31")
val_end = pd.Timestamp("2025-12-31")

# PRICE
price_train = price[
    price["Date"] <= train_end
].copy()

price_val = price[
    (price["Date"] > train_end) &
    (price["Date"] <= val_end)
].copy()

price_test = price[
    price["Date"] > val_end
].copy()

# RETURN
return_train = returns[
    returns["Date"] <= train_end
].copy()

return_val = returns[
    (returns["Date"] > train_end) &
    (returns["Date"] <= val_end)
].copy()

return_test = returns[
    returns["Date"] > val_end
].copy()


In [4]:
# ============================================================
# 4. IN KÍCH THƯỚC VÀ KHOẢNG THỜI GIAN
# ============================================================

def show_split(name, df):
    print(
        f"{name:<12}: {df.shape} | "
        f"{df['Date'].min().date()} -> {df['Date'].max().date()}"
        if len(df) > 0 else
        f"{name:<12}: {df.shape} | rỗng"
    )

print("PRICE")
show_split("Train", price_train)
show_split("Validation", price_val)
show_split("Test", price_test)

print("\nRETURN")
show_split("Train", return_train)
show_split("Validation", return_val)
show_split("Test", return_test)


PRICE
Train       : (2347, 6) | 2016-01-04 -> 2024-12-31
Validation  : (262, 6) | 2025-01-01 -> 2025-12-31
Test        : (175, 6) | 2026-01-01 -> 2026-08-31

RETURN
Train       : (2346, 7) | 2016-01-05 -> 2024-12-31
Validation  : (262, 7) | 2025-01-01 -> 2025-12-31
Test        : (175, 7) | 2026-01-01 -> 2026-08-31


In [5]:
# ============================================================
# 5. KIỂM TRA KHÔNG CÓ GIAO NHAU GIỮA CÁC TẬP
# ============================================================

assert price_train["Date"].max() < price_val["Date"].min()
assert price_val["Date"].max() < price_test["Date"].min()

assert return_train["Date"].max() < return_val["Date"].min()
assert return_val["Date"].max() < return_test["Date"].min()

print("Ranh giới thời gian hợp lệ.")


Ranh giới thời gian hợp lệ.


In [6]:
# ============================================================
# 6. LƯU
# ============================================================

price_train.to_csv(PROCESSED_DIR / "price_train.csv", index=False)
price_val.to_csv(PROCESSED_DIR / "price_validation.csv", index=False)
price_test.to_csv(PROCESSED_DIR / "price_test.csv", index=False)

return_train.to_csv(PROCESSED_DIR / "return_train.csv", index=False)
return_val.to_csv(PROCESSED_DIR / "return_validation.csv", index=False)
return_test.to_csv(PROCESSED_DIR / "return_test.csv", index=False)

print("\nĐã lưu toàn bộ Train / Validation / Test.")



Đã lưu toàn bộ Train / Validation / Test.
